# Split collapsed candidate-bank DPO pairs into the three-stage curriculum

Read `dpo_pairs.jsonl` from the collapsed C3/C2/C1 candidate-bank judgments and create the staged curriculum files:

- Stage 1 — Concept Learning: `C3 > C1`
- Stage 2 — Format Alignment: `C3 > C2`
- Stage 3 — Hierarchical Ranking: `C2 > C1`

The class scheme is `C3 > C2 > C1`, where C1 combines the old related-incorrect and fully-incorrect classes.


In [ ]:
from pathlib import Path
import json
from collections import Counter, defaultdict

PAIR_FILE = Path(r"/home/dinh-tuan/Documents/Project/BioASQ_MRES/Task-Structured Counterfactual Preference Mining/Artifacts/cse_dpo/class_judgments/candidate_banks_train_first400_c3_c1_v1_gold_c3/dpo_pairs.jsonl")
OUTPUT_ROOT = Path(r"/home/dinh-tuan/Documents/Project/BioASQ_MRES/Task-Structured Counterfactual Preference Mining/Artifacts/cse_dpo/class_judgments/candidate_banks_train_first400_c3_c1_v1_gold_c3/staged_curriculum_pairs")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
rows = [json.loads(line) for line in PAIR_FILE.open(encoding='utf-8') if line.strip()]
len(rows), PAIR_FILE


In [ ]:
stage_specs = [
    (
        "concept_learning",
        "dpo_stage1_concept_learning_all_pairs.jsonl",
        "Stage 1 — Concept Learning: C3 > C1",
        lambda r: r.get("chosen_class") == "C3" and r.get("rejected_class") == "C1",
    ),
    (
        "format_alignment",
        "dpo_stage2_format_alignment_all_pairs.jsonl",
        "Stage 2 — Format Alignment: C3 > C2",
        lambda r: r.get("chosen_class") == "C3" and r.get("rejected_class") == "C2",
    ),
    (
        "hierarchical_ranking",
        "dpo_stage3_hierarchical_ranking_all_pairs.jsonl",
        "Stage 3 — Hierarchical Ranking: C2 > C1",
        lambda r: r.get("chosen_class") == "C2" and r.get("rejected_class") == "C1",
    ),
]

summary = {
    "source_pairs": str(PAIR_FILE),
    "class_scheme": "C3_C2_C1_collapsed_from_C3_C2_C1_C0",
    "curriculum": {},
}

for stage_name, filename, stage_rule, predicate in stage_specs:
    selected = []
    for row in rows:
        if predicate(row):
            out = dict(row)
            out["stage"] = stage_name
            out["stage_rule"] = stage_rule
            selected.append(out)
    path = OUTPUT_ROOT / filename
    with path.open('w', encoding='utf-8') as f:
        for row in selected:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')
    summary["curriculum"][stage_name] = {
        "stage_rule": stage_rule,
        "pair_count": len(selected),
        "question_count": len({r.get("question_id") for r in selected}),
        "pair_class_counts": dict(Counter(f"{r.get('chosen_class')}>{r.get('rejected_class')}" for r in selected)),
        "file": str(path),
    }

(OUTPUT_ROOT / 'summary.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')
print(json.dumps(summary, indent=2, ensure_ascii=False))
